In [2]:
# class for icd code hierarchial tree
class ICDcodeNode:
    def __init__(self, icd_code, description, children, parent):
        self.icd_code = icd_code
        self.description = description
        self.children  =  children
        self.parent = parent

    def get_children(self):
        return self.children
    
    def add_child(self, child):
        if child not in self.children:
            self.children.append(child)

    def get_parent(self):
        return self.parent
    
    def set_parent(self,parent):
        self.parent = parent

    def __repr__(self):
        return f'{self.icd_code} - {self.description} -  Children: {[x.icd_code for x in self.children]}'

In [15]:

import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pickle
import networkx as nx
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# ===========================
# CONFIG
# ===========================
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BILORD_MODEL = "FremyCompany/BioLORD-2023"

SAPBERT_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_faiss_sapbert_from_pubmedbert.index"
SAPBERT_META = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_metadata_sapbert_from_pubmedbert.xlsx"
BILORD_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_faiss_biolord.index"
BILORD_META = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_metadata_biolord.xlsx"
ICD_GRAPH_PICKLE = r"C:\\Users\\UNegi\\Documents\\Project\\makethon\\icd_code_hierarchy.pkl"

# ===========================
# LOAD MODELS & INDEXES
# ===========================
sap_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
sap_model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE).eval()

bilord_tokenizer = AutoTokenizer.from_pretrained(BILORD_MODEL)
bilord_model = AutoModel.from_pretrained(BILORD_MODEL).to(DEVICE).eval()

sap_index = faiss.read_index(SAPBERT_INDEX)
bilord_index = faiss.read_index(BILORD_INDEX)

sap_meta_df = pd.read_excel(SAPBERT_META)
bilord_meta_df = pd.read_excel(BILORD_META)

sap_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in sap_meta_df.iterrows()}
bilord_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in bilord_meta_df.iterrows()}

# ===========================
# EMBEDDING FUNCTIONS
# ===========================
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

def embed_texts(texts, tokenizer, model):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)  # L2 normalization
    return embs

# ===========================
# SEARCH FUNCTION
# ===========================
def search_model(query, index, tokenizer, model, metadata, top_k=200):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = metadata[idx]
        results.append({"code": meta["code"], "description": meta["description"], "score": float(score), "id": idx})
    return results

# ===========================
# LOAD ICD GRAPH
# ===========================
with open(ICD_GRAPH_PICKLE, "rb") as f:
    node_dict = pickle.load(f)
root_node = node_dict['root']
graph = nx.Graph()

def build_networkx_graph(node):
    graph.add_node(node.icd_code, description=node.description)
    for child in node.get_children():
        graph.add_edge(node.icd_code, child.icd_code)
        build_networkx_graph(child)

build_networkx_graph(root_node)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# ===========================
# LATE FUSION WITH BOOSTING
# ===========================
def late_fusion_with_graph_proximity(query, alpha=1, beta=0, gamma=0, overlap_bonus=0.1, top_k=200):
    sap_results = search_model(query, sap_index, sap_tokenizer, sap_model, sap_metadata, top_k)
    bilord_results = search_model(query, bilord_index, bilord_tokenizer, bilord_model, bilord_metadata, top_k)

    combined = {}
    for r in sap_results:
        combined[r["code"]] = alpha * r["score"]

    for r in bilord_results:
        if r["code"] in combined:
            combined[r["code"]] += beta * r["score"] + overlap_bonus  # extra boost for overlap
        else:
            combined[r["code"]] = beta * r["score"]

    ranked_initial = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    anchor_code = ranked_initial[0][0]

    # Graph-based boosting
    boosted_nodes = {}
    if anchor_code in graph:
        boosted_nodes[anchor_code] = gamma * 1.0
        for parent in graph.neighbors(anchor_code):
            boosted_nodes[parent] = gamma * 0.8
            for sib in graph.neighbors(parent):
                if sib != anchor_code:
                    boosted_nodes[sib] = gamma * 0.6
            for gp in graph.neighbors(parent):
                if gp != anchor_code and gp != parent:
                    boosted_nodes[gp] = gamma * 0.7
                    for block_child in graph.neighbors(gp):
                        if block_child != parent:
                            boosted_nodes[block_child] = gamma * 0.6

    for code, boost in boosted_nodes.items():
        if code in combined:
            combined[code] += boost

    # Normalize to 0–100 for display
    scores = list(combined.values())
    min_score, max_score = min(scores), max(scores)
    normalized_combined = {code: 100 * (score - min_score) / (max_score - min_score) for code, score in combined.items()}

    ranked_final = sorted(normalized_combined.items(), key=lambda x: x[1], reverse=True)

    formatted_results = []
    for code, score in ranked_final:
        description = sap_meta_df.loc[sap_meta_df["Code"] == code, "Description"].values
        if len(description) == 0:
            description = bilord_meta_df.loc[bilord_meta_df["Code"] == code, "Description"].values
        description = description[0] if len(description) > 0 else "Description not found"
        formatted_results.append({
            "ICD_Code": code.strip(),
            "Description": description,
            "Score (0-100)": round(score, 2)
        })

    return formatted_results, anchor_code, boosted_nodes

# ===========================
# VISUALIZATION
# ===========================
def visualize_boosted_graph(anchor_code, boosted_nodes):
    pos = nx.spring_layout(graph, seed=42)
    node_colors = []
    node_sizes = []
    for node in graph.nodes():
        if node == anchor_code:
            node_colors.append("red")
            node_sizes.append(900)
        elif node in boosted_nodes:
            node_colors.append("orange" if boosted_nodes[node] >= 0.7 else "yellow")
            node_sizes.append(700)
        else:
            node_colors.append("lightgray")
            node_sizes.append(400)
    plt.figure(figsize=(12, 8))
    nx.draw(graph, pos, with_labels=True, node_color=node_colors, node_size=node_sizes, font_size=8)
    plt.title(f"ICD Graph Boost Visualization (Anchor: {anchor_code})")
    plt.show()

# ===========================
# TEST PIPELINE
# ===========================
query = "Heart Attack"
final_results, anchor_code, boosted_nodes = late_fusion_with_graph_proximity(query)
print("Top 10 ICD codes after late fusion + hierarchy boost:")
for res in final_results[:30]:
    print(res)


Graph loaded: 97802 nodes, 97831 edges
Top 10 ICD codes after late fusion + hierarchy boost:
{'ICD_Code': 'I249', 'Description': 'Acute ischemic heart disease, unspecified', 'Score (0-100)': 100.0}
{'ICD_Code': 'I209', 'Description': 'Angina pectoris, unspecified', 'Score (0-100)': 97.5}
{'ICD_Code': 'I469', 'Description': 'Cardiac arrest, cause unspecified', 'Score (0-100)': 96.9}
{'ICD_Code': 'I219', 'Description': 'Acute myocardial infarction, unspecified', 'Score (0-100)': 92.78}
{'ICD_Code': 'I519', 'Description': 'Heart disease, unspecified', 'Score (0-100)': 89.78}
{'ICD_Code': 'I200', 'Description': 'Unstable angina', 'Score (0-100)': 87.31}
{'ICD_Code': 'I2089', 'Description': 'Other forms of angina pectoris', 'Score (0-100)': 87.03}
{'ICD_Code': 'R002', 'Description': 'Palpitations', 'Score (0-100)': 86.55}
{'ICD_Code': 'Z8674', 'Description': 'Personal history of sudden cardiac arrest', 'Score (0-100)': 86.31}
{'ICD_Code': 'I2489', 'Description': 'Other forms of acute ischem

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def build_networkx_graph(root_node):
    """Convert ICDcodeNode tree to NetworkX graph"""
    graph = nx.Graph()

    def add_nodes_edges(node):
        # Add node with its ICD code and description as attributes
        graph.add_node(node.icd_code, description=node.description)

        # Add edges for each child
        for child in node.get_children():
            graph.add_edge(node.icd_code, child.icd_code)
            add_nodes_edges(child)

    add_nodes_edges(root_node)
    return graph

def visualize_large_graph(graph):
    """Visualize NetworkX graph with matplotlib"""
    pos = nx.spring_layout(graph, k=0.1, iterations=100) #spacing between nodes

    plt.figure(figsize=(8.5, 8.5))

    # Draw edges
    nx.draw_networkx_edges(graph, pos, alpha=0.5, width=0.5, edge_color='#888')

    # Draw nodes
    node_sizes = []
    for node in graph.nodes():
        # Size inversely proportional to ICD code length (longer codes are lower in the hierarchy)
        code_length = len(node)
        if '-' in node: #(code is a section/chapter)
            code_length = 1
        node_sizes.append(1 / (code_length + 1) * 2000)  # Scale factor

    nx.draw_networkx_nodes(
        graph, pos, 
        node_size=node_sizes,
        node_color=[len(graph.edges(n)) for n in graph.nodes()],
        cmap=plt.cm.plasma, alpha=0.5
    )

    # Add labels to nodes (ICD codes)
    labels = {node: node for node in graph.nodes()}
    nx.draw_networkx_labels(graph, pos, labels, font_size=8)

    plt.title('ICD Code Hierarchy Visualization', fontsize=16)
    plt.axis('off') 
    plt.tight_layout()
    plt.show()

with open(ICD_GRAPH_PICKLE, "rb") as f:
    node_dict = pickle.load(f)
root_node = node_dict['J40-J4A']

# Convert to NetworkX graph and visualize
graph = build_networkx_graph(root_node)
print('Built graph')
visualize_large_graph(graph)

In [ ]:
# class for icd code hierarchial tree
class ICDcodeNode:
    def __init__(self, icd_code, description, children, parent):
        self.icd_code = icd_code
        self.description = description
        self.children  =  children
        self.parent = parent

    def get_children(self):
        return self.children
    
    def add_child(self, child):
        if child not in self.children:
            self.children.append(child)

    def get_parent(self):
        return self.parent
    
    def set_parent(self,parent):
        self.parent = parent

    def __repr__(self):
        return f'{self.icd_code} - {self.description} -  Children: {[x.icd_code for x in self.children]}'
    

with open(ICD_GRAPH_PICKLE, "rb") as f:
    node_dict = pickle.load(f)
root_node = node_dict['root']
graph = nx.Graph()

def build_networkx_graph(node):
    graph.add_node(node.icd_code, description=node.description)
    for child in node.get_children():
        graph.add_edge(node.icd_code, child.icd_code)
        build_networkx_graph(child)

build_networkx_graph(root_node)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

Graph loaded: 97802 nodes, 97831 edges


In [ ]:

import pandas as pd
import faiss
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import pickle
import networkx as nx
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32

# ===========================
# CONFIG
# ===========================
SAPBERT_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BILORD_MODEL = "FremyCompany/BioLORD-2023"

SAPBERT_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_faiss_sapbert_from_pubmedbert.index"
SAPBERT_META = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_metadata_sapbert_from_pubmedbert.xlsx"
BILORD_INDEX = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_faiss_biolord.index"
BILORD_META = r"C:\Users\UNegi\Documents\Project\makethon\index\icd10_metadata_biolord.xlsx"
ICD_GRAPH_PICKLE = r"C:\\Users\\UNegi\\Documents\\Project\\makethon\\icd_code_hierarchy.pkl"

# ===========================
# LOAD MODELS & INDEXES
# ===========================
sap_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL)
sap_model = AutoModel.from_pretrained(SAPBERT_MODEL).to(DEVICE).eval()

bilord_tokenizer = AutoTokenizer.from_pretrained(BILORD_MODEL)
bilord_model = AutoModel.from_pretrained(BILORD_MODEL).to(DEVICE).eval()

sap_index = faiss.read_index(SAPBERT_INDEX)
bilord_index = faiss.read_index(BILORD_INDEX)

sap_meta_df = pd.read_excel(SAPBERT_META)
bilord_meta_df = pd.read_excel(BILORD_META)

sap_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in sap_meta_df.iterrows()}
bilord_metadata = {i: {"code": row["Code"], "description": row["Description"]} for i, row in bilord_meta_df.iterrows()}

# ===========================
# EMBEDDING FUNCTIONS
# ===========================
def mean_pooling(last_hidden_state, attention_mask):
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return (last_hidden_state * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1e-9)

def embed_texts(texts, tokenizer, model):
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), BATCH_SIZE):
            batch = texts[i:i+BATCH_SIZE]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            out = model(**enc)
            emb = mean_pooling(out.last_hidden_state, enc["attention_mask"])
            emb = emb.cpu().numpy()
            all_embs.append(emb)
    embs = np.vstack(all_embs).astype("float32")
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)  # L2 normalization
    return embs

def softmax_normalize(scores):
    arr = np.array(scores)
    exp_arr = np.exp(arr - np.max(arr))  # stability trick
    return exp_arr / (exp_arr.sum() + 1e-8)

# ===========================
# SEARCH FUNCTION
# ===========================
def search_model(query, index, tokenizer, model, metadata, top_k=100):
    q_emb = embed_texts([query], tokenizer, model)
    D, I = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        meta = metadata[idx]
        results.append({"code": meta["code"], "description": meta["description"], "score": float(score), "id": idx})
    return results

# ===========================
# LOAD ICD GRAPH
# ===========================
with open(ICD_GRAPH_PICKLE, "rb") as f:
    node_dict = pickle.load(f)
root_node = node_dict['root']
graph = nx.Graph()

def build_networkx_graph(node):
    graph.add_node(node.icd_code, description=node.description)
    for child in node.get_children():
        graph.add_edge(node.icd_code, child.icd_code)
        build_networkx_graph(child)

build_networkx_graph(root_node)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

# ===========================
# BOOSTING LOGIC
# ===========================
def late_fusion_with_graph_proximity(query, alpha=0.5, beta=0.5, gamma=0.3, top_k=100):
    sap_results = search_model(query, sap_index, sap_tokenizer, sap_model, sap_metadata, top_k)
    bilord_results = search_model(query, bilord_index, bilord_tokenizer, bilord_model, bilord_metadata, top_k)

    sap_norm = softmax_normalize([r["score"] for r in sap_results])
    bilord_norm = softmax_normalize([r["score"] for r in bilord_results])
    for i, r in enumerate(sap_results): r["norm_score"] = sap_norm[i]
    for i, r in enumerate(bilord_results): r["norm_score"] = bilord_norm[i]

    combined = {}
    for r in sap_results:
        combined[r["code"]] = alpha * r["norm_score"]
    for r in bilord_results:
        combined[r["code"]] = combined.get(r["code"], 0) + beta * r["norm_score"]

    ranked_initial = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    anchor_code = ranked_initial[0][0]

    boosted_nodes = {}
    if anchor_code in graph:
        boosted_nodes[anchor_code] = gamma * 1.0
        for parent in graph.neighbors(anchor_code):
            boosted_nodes[parent] = gamma * 0.8
            for sib in graph.neighbors(parent):
                if sib != anchor_code:
                    boosted_nodes[sib] = gamma * 0.6
            for gp in graph.neighbors(parent):
                if gp != anchor_code and gp != parent:
                    boosted_nodes[gp] = gamma * 0.7
                    for block_child in graph.neighbors(gp):
                        if block_child != parent:
                            boosted_nodes[block_child] = gamma * 0.6

    for code, boost in boosted_nodes.items():
        if code in combined:
            combined[code] += boost

    scores = list(combined.values())
    min_score, max_score = min(scores), max(scores)
    normalized_combined = {code: 100 * (score - min_score) / (max_score - min_score) for code, score in combined.items()}

    ranked_final = sorted(normalized_combined.items(), key=lambda x: x[1], reverse=True)

    formatted_results = []
    for code, score in ranked_final:
        description = sap_meta_df.loc[sap_meta_df["Code"] == code, "Description"].values
        if len(description) == 0:
            description = bilord_meta_df.loc[bilord_meta_df["Code"] == code, "Description"].values
        description = description[0] if len(description) > 0 else "Description not found"
        formatted_results.append({
            "ICD_Code": code.strip(),
            "Description": description,
            "Score (0-100)": round(score, 2)
        })

    return sap_results ,bilord_results,combined,formatted_results, anchor_code, boosted_nodes

# ===========================
# VISUALIZATION
# ===========================

# ===========================
# TEST PIPELINE
# ===========================
query = "COPD"
sap_results ,bilord_results,combined,formatted_results, anchor_code, boosted_nodes = late_fusion_with_graph_proximity(query)
print("Top 10 ICD codes after late fusion + hierarchy boost:")
for res in final_results[:10]:
    print(res)



Graph loaded: 97802 nodes, 97831 edges
Top 10 ICD codes after late fusion + hierarchy boost:
{'ICD_Code': 'J449', 'Description': 'Chronic obstructive pulmonary disease, unspecified', 'Score (0-100)': np.float64(100.0)}
{'ICD_Code': 'J4489', 'Description': 'Other specified chronic obstructive pulmonary disease', 'Score (0-100)': np.float64(87.51)}
{'ICD_Code': 'J849', 'Description': 'Interstitial pulmonary disease, unspecified', 'Score (0-100)': np.float64(81.03)}
{'ICD_Code': 'J984', 'Description': 'Other disorders of lung', 'Score (0-100)': np.float64(80.41)}
{'ICD_Code': 'J441', 'Description': 'Chronic obstructive pulmonary disease with (acute) exacerbation', 'Score (0-100)': np.float64(78.61)}
{'ICD_Code': 'J42', 'Description': 'Unspecified chronic bronchitis', 'Score (0-100)': np.float64(78.54)}
{'ICD_Code': 'J64', 'Description': 'Unspecified pneumoconiosis', 'Score (0-100)': np.float64(77.16)}
{'ICD_Code': 'J8410', 'Description': 'Pulmonary fibrosis, unspecified', 'Score (0-100)':

In [11]:
sap_results[:10]

[{'code': 'J449    ',
  'description': 'Chronic obstructive pulmonary disease, unspecified',
  'score': 0.7842878699302673,
  'id': np.int64(10596),
  'norm_score': np.float64(0.012551457205979869)},
 {'code': 'J849    ',
  'description': 'Interstitial pulmonary disease, unspecified',
  'score': 0.6808487772941589,
  'id': np.int64(10696),
  'norm_score': np.float64(0.011318037343928701)},
 {'code': 'J64     ',
  'description': 'Unspecified pneumoconiosis',
  'score': 0.6369710564613342,
  'id': np.int64(10632),
  'norm_score': np.float64(0.010832165094247708)},
 {'code': 'J410    ',
  'description': 'Simple chronic bronchitis',
  'score': 0.6229656338691711,
  'id': np.int64(10583),
  'norm_score': np.float64(0.010681513476839245)},
 {'code': 'J4489   ',
  'description': 'Other specified chronic obstructive pulmonary disease',
  'score': 0.6221534013748169,
  'id': np.int64(10595),
  'norm_score': np.float64(0.010672841126963439)},
 {'code': 'Q336    ',
  'description': 'Congenital hy

In [12]:
bilord_results[:10]

[{'code': 'J449    ',
  'description': 'Chronic obstructive pulmonary disease, unspecified',
  'score': 0.8114243745803833,
  'id': np.int64(10596),
  'norm_score': np.float64(0.01221853078059358)},
 {'code': 'J4489   ',
  'description': 'Other specified chronic obstructive pulmonary disease',
  'score': 0.8066036701202393,
  'id': np.int64(10595),
  'norm_score': np.float64(0.012159770601287272)},
 {'code': 'J984    ',
  'description': 'Other disorders of lung',
  'score': 0.7954512238502502,
  'id': np.int64(10770),
  'norm_score': np.float64(0.012024912807913436)},
 {'code': 'J988    ',
  'description': 'Other specified respiratory disorders',
  'score': 0.7758183479309082,
  'id': np.int64(10774),
  'norm_score': np.float64(0.01179113159478515)},
 {'code': 'J989    ',
  'description': 'Respiratory disorder, unspecified',
  'score': 0.7407017350196838,
  'id': np.int64(10775),
  'norm_score': np.float64(0.011384252903449276)},
 {'code': 'Z8709   ',
  'description': 'Personal history

In [14]:
combined

{'J449    ': np.float64(0.012384993993286724),
 'J849    ': np.float64(0.010913415391479409),
 'J64     ': np.float64(0.010613348098484332),
 'J410    ': np.float64(0.01047112622588796),
 'J4489   ': np.float64(0.011416305864125356),
 'Q336    ': np.float64(0.005326092799970458),
 'I279    ': np.float64(0.010357644249614033),
 'J84117  ': np.float64(0.005298447059875744),
 'J441    ': np.float64(0.010725893323044106),
 'J84115  ': np.float64(0.010207321701017371),
 'J42     ': np.float64(0.010720507839603323),
 'I2781   ': np.float64(0.00519037248380245),
 'J45991  ': np.float64(0.009815014963086345),
 'P279    ': np.float64(0.009850636776902165),
 'F1820   ': np.float64(0.0051276284177932336),
 'J677    ': np.float64(0.0102551424716),
 'E803    ': np.float64(0.005116557614112975),
 'J479    ': np.float64(0.005109891947810106),
 'J40     ': np.float64(0.010182956061725656),
 'J983    ': np.float64(0.010161394467400022),
 'P2811   ': np.float64(0.00509012024188704),
 'J84111  ': np.floa